# BACKTESTING_272 — Stratégie Momentum Dual Equal Risk contribution (Risk Parity)
 
**Univers** : STOXX Europe 600  
**Période** : 2016-12-30 → 2026-02-20  
**Benchmark** : ESTRON Index


## 1. Importation des Libraries <a id='1'></a>

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

import pandas as pd
import numpy as np
from IPython.display import display


from backtesting import DataLoader, BacktestEngine, ReportingEngine


## 2. Paramètres du portefeuille <a id='1'></a>

In [2]:
INITIAL_CAPITAL = 1_000_000

TRANSACTION_COST = 0.0005

SELECTION_QUANTILE = 0.40

METHODS = ["risk_parity"]

"""
- SECTOR_MIN : nombre minimum de poids par secteur
- SECTOR_MAX : nombre maximum de poids par secteur
"""
SECTOR_MIN = None
SECTOR_MAX = None

print(f"Capital initial   : {INITIAL_CAPITAL:,.0f} €")
print(f"Frais transaction : {TRANSACTION_COST * 10_000:.1f} bps")
print(f"Sélection         : top/bottom {SELECTION_QUANTILE*100:.0f}%")
print(f"Méthodes          : {METHODS}")
print(f"Contraintes       : SECTOR_MIN={SECTOR_MIN}, SECTOR_MAX={SECTOR_MAX}")

Capital initial   : 1,000,000 €
Frais transaction : 5.0 bps
Sélection         : top/bottom 40%
Méthodes          : ['risk_parity']
Contraintes       : SECTOR_MIN=None, SECTOR_MAX=None



## 3. Chargement des données <a id='1'></a>

In [3]:
"""
- start_date : date de début du backtest (format "YYYY-MM-DD")
- end_date : date de fin du backtest (format "YYYY-MM-DD")
"""

loader = DataLoader(start_date="2016-12-30", end_date="2026-02-20")

"""
- initial_capital    : implémente le capital initinial du portefeuille
- transaction_cost   : implémente le coût de transaction par trade 
- selection_quantile : implémente le quantile utilisé pour sélectionner les positions long/short au travers des signaux
- sector_min         : implémente le poids minimum autorisé par secteur (contrainte de diversification)
- sector_max         : implémente le poids maximum autorisé par secteur (contrainte de diversification)
"""

engine = BacktestEngine(
    data_loader=loader,
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=TRANSACTION_COST,
    selection_quantile=SELECTION_QUANTILE,
    sector_min=SECTOR_MIN,
    sector_max=SECTOR_MAX,
)

results = {}

print(f"\nDonnées chargées : {len(loader.rebalancing_dates)} dates de rebalancement")
print(f"Tickers disponibles : {loader.prices.shape[1]}")
print(f"Période des prix : {loader.prices.index[0].date()} → {loader.prices.index[-1].date()}")

[DataLoader] Chargement des données en cours...
[DataLoader] Prêt : 111 dates de rebalancement (2016-12-30 → 2026-02-20).

Données chargées : 111 dates de rebalancement
Tickers disponibles : 980
Période des prix : 2015-09-01 → 2026-02-20



## 4. Backtesting <a id='1'></a>

In [4]:
"""
- Choix : risk_parity 
"""
METHOD = "risk_parity"

if METHOD in METHODS:
    print(f"{'='*60}")
    print(f"  Lancement : {METHOD}")
    print(f"{'='*60}")
    results[METHOD] = engine.run(METHOD)
    nav_final = results[METHOD].nav.iloc[-1]
    total_ret = nav_final / INITIAL_CAPITAL - 1
    print(f"\n  NAV finale : {nav_final:,.0f} €  |  Rendement : {total_ret:.2%}")
else:
    print(f"'{METHOD}' non sélectionné dans METHODS — cellule ignorée.")

  Lancement : risk_parity

[BacktestEngine] Démarrage — méthode : risk_parity
  [  1/111] 2016-12-30 NAV=   1,009,940€  Long=232  Short=232
  [  2/111] 2017-01-31 NAV=     998,538€  Long=230  Short=230
  [  3/111] 2017-02-28 NAV=   1,001,636€  Long=231  Short=231
  [  4/111] 2017-03-31 NAV=   1,009,764€  Long=230  Short=230
  [  5/111] 2017-04-28 NAV=   1,003,963€  Long=230  Short=230
  [  6/111] 2017-05-31 NAV=   1,012,584€  Long=229  Short=229
  [  7/111] 2017-06-30 NAV=   1,026,400€  Long=231  Short=231
  [  8/111] 2017-07-31 NAV=   1,047,994€  Long=231  Short=231
  [  9/111] 2017-08-31 NAV=   1,045,595€  Long=231  Short=231
  [ 10/111] 2017-09-29 NAV=   1,062,560€  Long=230  Short=230
  [ 11/111] 2017-10-31 NAV=   1,071,332€  Long=203  Short=203
  [ 12/111] 2017-11-30 NAV=   1,065,304€  Long=232  Short=232
  [ 13/111] 2017-12-29 NAV=   1,082,842€  Long=231  Short=231
  [ 14/111] 2018-01-31 NAV=   1,070,960€  Long=232  Short=232
  [ 15/111] 2018-02-28 NAV=   1,072,990€  Long=167  Sh


## 5. Reporting <a id='1'></a>

In [5]:
METHOD_TO_REPORT = METHODS.pop(0) 

"""
- AS_OF_DATE       : date d’arrêté des métriques de performance (format "YYYY-MM-DD")
"""
AS_OF_DATE = "2026-01-30"

reporter = ReportingEngine(results[METHOD_TO_REPORT], loader)
as_of = pd.Timestamp(AS_OF_DATE)

In [6]:

metrics = reporter.compute_metrics(as_of)


def format_metric(v, key=None):
    """
    - si la valeur est "--", elle est retournée telle qu'une valeur manquante,
    - si la valeur est un float :
         si la métrique est un ratio (Sharpe, IR, Corrélation, etc.) -> format décimal
         sinon (rendements, vol, drawdown, TC%) -> format en %
    - sinon, la valeur est convertie en chaîne de caractères.
    """
    if v == "--":
        return "--"

    if isinstance(v, float):

        if key is not None and ("(€)" in key or "€" in key):
            return f"{v:,.2f}"


        ratio_keys = (
            "Sharpe", "Sortino", "Information Ratio",
            "Calmar", "Corrélation", "Tracking Error"
        )
        if key is not None and any(rk in key for rk in ratio_keys):
            return f"{v:.4f}"


        return f"{v*100:.2f}%"

    return str(v)


"""
- Serie pandas indéxée par le nom des indicateurs et datée à la date de reporting
"""

metrics_series = pd.Series({
    k: format_metric(v, k) for k, v in metrics.items()
}, name=str(as_of.date()))
metrics_series.index.name = "Métrique"

categories = {
    "Performance": [
        "Rendement cumulé (période)", "Rendement cumulé 1 an",
        "Rendement cumulé 2 ans", "Rendement cumulé YTD", "Rendement cumulé MTD",
        "Rendement annualisé (période)",
        "CAGR (période)", "CAGR 1 an", "CAGR 2 ans", "CAGR YTD", "CAGR MTD",
    ]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Performance
──────────────────────────────────────────────────
  Rendement cumulé (période)                          44.08%
  Rendement cumulé 1 an                               10.92%
  Rendement cumulé 2 ans                              15.88%
  Rendement cumulé YTD                                 2.42%
  Rendement cumulé MTD                                 2.42%
  Rendement annualisé (période)                        4.03%
  CAGR (période)                                       3.54%
  CAGR 1 an                                           11.22%
  CAGR 2 ans                                           8.30%
  CAGR YTD                                            33.85%
  CAGR MTD                                            33.85%


In [7]:
categories = {
    "Risque & Ratios": [
        "Sharpe ratio (période)", "Sharpe ratio 1 an",
        "Sortino ratio (période)", "Sortino ratio 1 an",
        "Tracking Error (période)", "Tracking Error 1 an", "Tracking Error 3 ans",
        "Information Ratio (période)", "Information Ratio 1 an",
        "Information Ratio 2 ans",
        "Calmar ratio (période)",
        "VaR 95% (période)", "VaR 95% 1 an",
        "CVaR 95% (période)", "CVaR 95% 1 an",
        "Volatilité (période)", "Volatilité 1 an", "Volatilité 2 ans",
        "Max Drawdown (période)", "Max Drawdown 1 an",
        "Max Drawdown 2 ans", "Max Drawdown YTD", "Max Drawdown MTD",
    ]}


for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>20}")


──────────────────────────────────────────────────
  Risque & Ratios
──────────────────────────────────────────────────
  Sharpe ratio (période)                                      0.4022
  Sharpe ratio 1 an                                           1.1734
  Sortino ratio (période)                                     0.4441
  Sortino ratio 1 an                                          1.6484
  Tracking Error (période)                                    0.0910
  Tracking Error 1 an                                         0.0752
  Tracking Error 3 ans                                        0.0679
  Information Ratio (période)                                 0.4022
  Information Ratio 1 an                                      1.1734
  Information Ratio 2 ans                                     0.7936
  Calmar ratio (période)                                      0.2303
  VaR 95% (période)                                            0.84%
  VaR 95% 1 an                                     

In [8]:

categories = {
       "Benchmark (ESTER)": [
        "Corrélation avec benchmark (période)",
        "Benchmark - Rendement cumulé (période)",
        "Benchmark - Rendement cumulé 1 an",
        "Benchmark - Rendement cumulé 2 ans",
        "Benchmark - Rendement cumulé YTD",
        "Benchmark - Rendement cumulé MTD",
        "Benchmark - Rendement annualisé (période)",
    ],
    "Coûts de transaction": ["TC total (€)", "TC total (%)"]}

for cat, keys in categories.items():
    print(f"\n{'─'*50}")
    print(f"  {cat}")
    print(f"{'─'*50}")
    sub = {k: metrics.get(k, "--") for k in keys}
    for k, v in sub.items():
        val = format_metric(v, k)
        print(f"  {k:<45} {val:>12}")


──────────────────────────────────────────────────
  Benchmark (ESTER)
──────────────────────────────────────────────────
  Corrélation avec benchmark (période)               -0.0052
  Benchmark - Rendement cumulé (période)               8.38%
  Benchmark - Rendement cumulé 1 an                    2.15%
  Benchmark - Rendement cumulé 2 ans                   5.92%
  Benchmark - Rendement cumulé YTD                     0.16%
  Benchmark - Rendement cumulé MTD                     0.16%
  Benchmark - Rendement annualisé (période)            1.25%

──────────────────────────────────────────────────
  Coûts de transaction
──────────────────────────────────────────────────
  TC total (€)                                     51,807.11
  TC total (%)                                         5.18%


In [9]:
all_metrics_df = reporter.compute_all_metrics()
display(all_metrics_df.tail(5))

,Rendement cumulé (période),Rendement cumulé 1 an,Rendement cumulé 2 ans,Rendement cumulé YTD,Rendement cumulé MTD,Rendement annualisé (période),CAGR (période),CAGR 1 an,CAGR 2 ans,CAGR YTD,...,Corrélation avec benchmark (période),Benchmark - Rendement cumulé (période),Benchmark - Rendement cumulé 1 an,Benchmark - Rendement cumulé 2 ans,Benchmark - Rendement cumulé YTD,Benchmark - Rendement cumulé MTD,Benchmark - Rendement annualisé (période),TC total (€),TC total (%),PnL (€)
Date,,,,,,,,,,,,,,,,,,,,,
2025-10-31,0.360785,0.057027,0.083786,0.050427,0.002834,0.034806,0.03003,0.059914,0.044127,0.060891,...,-0.007921,0.078571,0.0244,0.064479,0.019022,0.00166,0.01222,49263.989811,0.049264,305273.800287
2025-11-28,0.381139,0.066996,0.099552,0.065891,0.014722,0.036189,0.031422,0.068596,0.055379,0.072725,...,-0.007203,0.080191,0.023416,0.062846,0.020552,0.001501,0.012304,49571.873398,0.049572,324489.783524
2025-12-31,0.405304,0.083885,0.157309,0.083885,0.016881,0.037816,0.033017,0.083489,0.079196,0.083945,...,-0.006396,0.082104,0.022359,0.061158,0.022359,0.001771,0.012421,50386.621173,0.050387,346849.031701
2026-01-30,0.440841,0.109174,0.158779,0.024233,0.024233,0.040274,0.035392,0.112232,0.083045,0.33846,...,-0.00519,0.083847,0.021518,0.059186,0.001611,0.001611,0.012511,51807.113094,0.051807,379487.343437
2026-02-20,0.464574,0.109683,0.171514,0.040861,0.016234,0.041838,0.036954,0.111207,0.088233,0.332177,...,-0.004385,0.085069,0.021045,0.057965,0.00274,0.001127,0.012569,52134.655283,0.052135,401882.311228



## 5. Graphiques <a id='1'></a>

### 5a. Rendements cumulés — Portefeuille vs Benchmark

In [10]:
fig_cumul = reporter.plot_cumulative_returns()
fig_cumul.show()

### 5b. Drawdowns du portefeuille

In [11]:
fig_dd = reporter.plot_drawdowns()
fig_dd.show()

### 5c. Volatilité glissante annualisée

In [12]:
fig_vol = reporter.plot_historical_volatility()
fig_vol.show()

### 5d. Corrélation glissante avec le Benchmark

In [13]:
fig_corr = reporter.plot_historical_correlation()
fig_corr.show()

### 5e. Composition du portefeuille

In [14]:

comp_date = pd.Timestamp(AS_OF_DATE)


comp_df = reporter.get_portfolio_composition(comp_date)
print(f"Portefeuille au {comp_date.date()} : {len(comp_df)} positions")
if comp_df.empty or 'Weight' not in comp_df.columns:
    print("  Composition indisponible pour cette date (probablement hors rebalancement).")
else:
    print(f"  Long  : {(comp_df['Weight'] > 0).sum()} positions")
    print(f"  Short : {(comp_df['Weight'] < 0).sum()} positions")
    print()
    display(comp_df.head())


Portefeuille au 2026-01-30 : 470 positions
  Long  : 235 positions
  Short : 235 positions



,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
0,ALLN SE Equity,ALLREAL HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,226.00,0.009558
1,MOBN SE Equity,MOBIMO HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,389.00,0.008907
2,SPSN SE Equity,SWISS PRIME SITE-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,131.30,0.008211
3,SCMN SE Equity,SWISSCOM AG-REG,SWITZERLAND,CHF,Services de communication,Services de télécommunicatio,633.50,0.007713
4,LI FP Equity,KLEPIERRE,FRANCE,EUR,Immobilier,SIIC au détail,32.44,0.007347


In [15]:
comp_charts = reporter.plot_composition_barcharts(comp_date)

comp_charts["Sector"].show()

In [16]:
comp_charts["Country"].show()

In [17]:
comp_charts["Industry"].show()

In [18]:
comp_charts["Currency"].show()

## 6. Analyse du portefeuille

### 6a. Top 10 positions 

In [19]:
top10_weights = reporter.get_top_10_weights(comp_date)

print(f"Top 10 positions au {comp_date.date()} :")
display(top10_weights)

Top 10 positions au 2026-01-30 :


,Ticker,Name,Sector,Country,Weight
0,ALLN SE Equity,ALLREAL HOLDING AG-REG,Immobilier,SWITZERLAND,0.009558
1,MOBN SE Equity,MOBIMO HOLDING AG-REG,Immobilier,SWITZERLAND,0.008907
2,SPSN SE Equity,SWISS PRIME SITE-REG,Immobilier,SWITZERLAND,0.008211
3,SCMN SE Equity,SWISSCOM AG-REG,Services de communication,SWITZERLAND,0.007713
4,LI FP Equity,KLEPIERRE,Immobilier,FRANCE,0.007347
5,IBE SQ Equity,IBERDROLA SA,Services aux collectivités,SPAIN,0.007149
6,GET FP Equity,GETLINK SE,Industrie,FRANCE,-0.007132
7,KPN NA Equity,KONINKLIJKE KPN NV,Services de communication,NETHERLANDS,0.007060
8,ENEL IM Equity,ENEL SPA,Services aux collectivités,ITALY,0.006851
9,PST IM Equity,POSTE ITALIANE SPA,Finance,ITALY,0.006844


### 6b. Top 10 contributions à la performance

In [20]:
top5_pos, top5_neg = reporter.get_top_10_return_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS (période {comp_date.date()}) :")
display(top5_pos)

print(f"Top 5 contributeurs NÉGATIFS (période {comp_date.date()}) :")
display(top5_neg)

Top 5 contributeurs POSITIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,ASML NA Equity,0.002154,ASML HOLDING NV,Technologies de l'information
1,SBMO NA Equity,0.001981,SBM OFFSHORE NV,Energie
2,EXPN LN Equity,0.001680,EXPERIAN PLC,Industrie
3,REL LN Equity,0.001606,RELX PLC,Industrie
4,ADM LN Equity,0.001582,ADMIRAL GROUP PLC,Finance


Top 5 contributeurs NÉGATIFS (période 2026-01-30) :


,Ticker,Contribution,Name,Sector
0,BEZ LN Equity,-0.002864,BEAZLEY PLC,Finance
1,DIE BB Equity,-0.002302,D'IETEREN GROUP,Consommation discrétionnaire
2,LOTB BB Equity,-0.002237,LOTUS BAKERIES,Consommation de base
3,ASM NA Equity,-0.002048,ASM INTERNATIONAL NV,Technologies de l'information
4,BN FP Equity,-0.001879,DANONE,Consommation de base


### 6c. Top 10 contributions au risque du portefeuille (MCTR)

In [21]:
top5_risk_pos, top5_risk_neg = reporter.get_top_10_risk_contribution(comp_date)

print(f"Top 5 contributeurs POSITIFS au risque au {comp_date.date()} :")
display(top5_risk_pos)

print(f"Top 5 contributeurs NÉGATIFS au risque au {comp_date.date()} :")
display(top5_risk_neg)

Top 5 contributeurs POSITIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,CABK SQ Equity,0.000558,0.004218,CAIXABANK SA,Finance
1,GLE FP Equity,0.000542,0.003403,SOCIETE GENERALE SA,Finance
2,UNI SQ Equity,0.000520,0.004200,UNICAJA BANCO SA,Finance
3,BAMI IM Equity,0.000516,0.004026,BANCO BPM SPA,Finance
4,HOT GY Equity,0.000505,0.003029,HOCHTIEF AG,Industrie


Top 5 contributeurs NÉGATIFS au risque au 2026-01-30 :


,Ticker,Risk Contribution,Weight,Name,Sector
0,DNB NO Equity,-0.000323,-0.005284,DNB BANK ASA,Finance
1,AGS BB Equity,-0.000315,-0.006247,AGEAS,Finance
2,BNP FP Equity,-0.000308,-0.003974,BNP PARIBAS,Finance
3,SPL PW Equity,-0.000277,-0.003182,SANTANDER BANK POLSKA SA,Finance
4,RBREW DC Equity,-0.000277,0.005624,ROYAL UNIBREW,Consommation de base


### 6d. Rendements cumulés calendaires du portefeuille

In [22]:
fig_calendar_returns = reporter.plot_calendar_returns_heatmap()

fig_calendar_returns.show()


## 7. Portfolio PnL

PnL cumulé du portefeuille

In [23]:
fig_pnl = reporter.plot_pnl()
fig_pnl.show()

## Exportation des Résultats

In [24]:
out_path = reporter.run_full_report(output_dir=None)
print("Reporting folder:", out_path)

out_dir = Path(out_path)

nav_files = sorted(out_dir.glob("nav_MomentumDual_*.parquet"))

metrics_files = sorted(out_dir.glob("metriques_MomentumDual_*.parquet"))

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))

bbu_files = sorted(out_dir.glob("bbu_MomentumDual_*.csv"))


[ReportingEngine] Génération du reporting dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_RiskParity_20161230_20260220
[Report] Métriques exportées : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_RiskParity_20161230_20260220/metriques_MomentumDual_RiskParity.parquet
[Report] BBU CSV exporté : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_RiskParity_20161230_20260220/bbu_MomentumDual_RiskParity.csv
[Report] Composition détaillée exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_RiskParity_20161230_20260220/composition_detaillee_MomentumDual_RiskParity.parquet
[Report] NAV exportée : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_RiskParity_20161230_20260220/nav_MomentumDual_RiskParity.parquet
[ReportingEngine] Reporting complet généré dans : /Users/arthurlenet/Desktop/Backtesting_M2272/data/stockage/MomentumDual_RiskParity_20161230_20260220

Repor

In [25]:

comp_files = sorted(out_dir.glob("composition_detaillee_MomentumDual_*.parquet"))
comp_df = pd.read_parquet(comp_files[0])
comp_df = pd.DataFrame(comp_df) 
comp_df["Date"] = pd.to_datetime(comp_df["Date"]) 
comp_files_df = comp_df[comp_df["Date"] == pd.Timestamp(AS_OF_DATE)] 
display(comp_files_df)

,Date,Ticker,Name,Country,Currency,Sector,Industry,Price,Weight
47280,2026-01-30,ALLN SE Equity,ALLREAL HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,226.000,0.009558
47281,2026-01-30,MOBN SE Equity,MOBIMO HOLDING AG-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,389.000,0.008907
47282,2026-01-30,SPSN SE Equity,SWISS PRIME SITE-REG,SWITZERLAND,CHF,Immobilier,Développement et gestion des,131.300,0.008211
47283,2026-01-30,SCMN SE Equity,SWISSCOM AG-REG,SWITZERLAND,CHF,Services de communication,Services de télécommunicatio,633.500,0.007713
47284,2026-01-30,LI FP Equity,KLEPIERRE,FRANCE,EUR,Immobilier,SIIC au détail,32.440,0.007347
...,...,...,...,...,...,...,...,...,...
47745,2026-01-30,BOL FP Equity,BOLLORE SE,FRANCE,EUR,Energie,"Pétrole, gaz et combustibles",4.814,-0.006544
47746,2026-01-30,TRYG DC Equity,TRYG A/S,DENMARK,DKK,Finance,Assurance,153.100,-0.006571
47747,2026-01-30,TRN IM Equity,TERNA-RETE ELETTRICA NAZIONA,ITALY,EUR,Services aux collectivités,Electricité,9.126,-0.006685
47748,2026-01-30,BCVN SE Equity,BANQUE CANTONALE VAUDOIS-REG,SWITZERLAND,CHF,Finance,Banques,104.600,-0.006786


In [26]:
bbu_df = pd.read_csv(bbu_files[0])
display(bbu_df.head())

,Date,Ticker,Weights
0,2016-12-30,KESKOB FH Equity,0.004990
1,2016-12-30,MRW LN Equity,0.004613
2,2016-12-30,TSCO LN Equity,0.003384
3,2016-12-30,MOWI NO Equity,0.005005
4,2016-12-30,JMT PL Equity,0.005002
